<a href="https://colab.research.google.com/github/TrollRider-Kristian/Springboard-AI-Mini-Projects/blob/main/Student_MLE_MiniProject_Fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini Project: Transfer Learning with Keras

Transfer learning is a machine learning technique where a model trained on one task is used as a starting point to solve a different but related task. Instead of training a model from scratch, transfer learning leverages the knowledge learned from the source task and applies it to the target task. This approach is especially useful when the target task has limited data or computational resources.

In transfer learning, the pre-trained model, also known as the "base model" or "source model," is typically trained on a large dataset and a more general problem (e.g., image classification on ImageNet, a vast dataset with millions of labeled images). The knowledge learned by the base model in the form of feature representations and weights captures common patterns and features in the data.

To perform transfer learning, the following steps are commonly followed:

1. Pre-training: The base model is trained on a source task using a large dataset, which can take a considerable amount of time and computational resources.

2. Feature Extraction: After pre-training, the base model is used as a feature extractor. The last few layers (classifier layers) of the model are discarded, and the remaining layers (feature extraction layers) are retained. These layers serve as feature extractors, producing meaningful representations of the data.

3. Fine-tuning: The feature extraction layers and sometimes some of the earlier layers are connected to a new set of layers, often called the "classifier layers" or "task-specific layers." These layers are randomly initialized, and the model is trained on the target task with a smaller dataset. The weights of the base model can be frozen during fine-tuning, or they can be allowed to be updated with a lower learning rate to fine-tune the model for the target task.

Transfer learning has several benefits:

1. Reduced training time and resource requirements: Since the base model has already learned generic features, transfer learning can save time and resources compared to training a model from scratch.

2. Improved generalization: Transfer learning helps the model generalize better to the target task, especially when the target dataset is small and dissimilar from the source dataset.

3. Better performance: By starting from a model that is already trained on a large dataset, transfer learning can lead to better performance on the target task, especially in scenarios with limited data.

4. Effective feature extraction: The feature extraction layers of the pre-trained model can serve as powerful feature extractors for different tasks, even when the task domains differ.

Transfer learning is commonly used in various domains, including computer vision, natural language processing (NLP), and speech recognition, where pre-trained models are fine-tuned for specific applications like object detection, sentiment analysis, or speech-to-text.

In this mini-project you will perform fine-tuning using Keras with a pre-trained VGG16 model on the CIFAR-10 dataset.

First, import all the libraries you'll need.

In [121]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Input, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

The CIFAR-10 dataset is a widely used benchmark dataset in the field of computer vision and machine learning. It stands for the "Canadian Institute for Advanced Research 10" dataset. CIFAR-10 was created by researchers at the CIFAR institute and was originally introduced as part of the Neural Information Processing Systems (NIPS) 2009 competition.

The dataset consists of 60,000 color images, each of size 32x32 pixels, belonging to ten different classes. Each class contains 6,000 images. The ten classes in CIFAR-10 are:

1. Airplane
2. Automobile
3. Bird
4. Cat
5. Deer
6. Dog
7. Frog
8. Horse
9. Ship
10. Truck

The images are evenly distributed across the classes, making CIFAR-10 a balanced dataset. The dataset is divided into two sets: a training set and a test set. The training set contains 50,000 images, while the test set contains the remaining 10,000 images.

CIFAR-10 is often used for tasks such as image classification, object recognition, and transfer learning experiments. The relatively small size of the images and the variety of classes make it a challenging dataset for training machine learning models, especially deep neural networks. It also serves as a good dataset for teaching and learning purposes due to its manageable size and straightforward class labels.

Here are your tasks:

1. Load the CIFAR-10 dataset after referencing the documentation [here](https://keras.io/api/datasets/cifar10/).
2. Normalize the pixel values so they're all in the range [0, 1].
3. Apply One Hot Encoding to the train and test labels using the [to_categorical](https://www.tensorflow.org/api_docs/python/tf/keras/utils/to_categorical) function.
4. Further split the the training data into training and validation sets using [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html). Use only 10% of the data for validation.  

In [88]:
# Load the CIFAR-10 dataset
# KRISTIAN_NOTE - The rgb_train and rgb_test variables consist of three 2D arrays
# each representing the 32x32 red, green, and blue values of each pixel, respectively.
# The label_train and lable_test sets consist of an array of integers corresponding
# to the index indicating each of the 10 classes.
(rgb_train, label_train), (rgb_test, label_test) = cifar10.load_data()

In [89]:
print ("There are: " + str(len(rgb_train)) + " rgb images in the training set.")
first = rgb_train[0][0][0]
print ("The first pixel in the first image has an rgb-value of: " + str(first))
print ("The normalized value of that pixel on a scale between 0 and 1 is: " + str(first / 255.0))

There are: 50000 rgb images in the training set.
The first pixel in the first image has an rgb-value of: [59 62 63]
The normalized value of that pixel on a scale between 0 and 1 is: [0.23137255 0.24313725 0.24705882]


In [90]:
# Normalize the pixel values to [0, 1]
def normalize_pixels_in_images (images):
  normalized_images = images.copy()
  for img in range(len(images)):
    for x in range(len(images[img])):
      for y in range(len(images[img][x])):
        normalized_images[img][x][y] = images[img][x][y] / 255
  return normalized_images

norm_rgb_train = normalize_pixels_in_images (rgb_train)
print(str(norm_rgb_train.shape) + " should be the same as that of the original: " + str(rgb_train.shape))
norm_rgb_test = normalize_pixels_in_images (rgb_test)
print(str(norm_rgb_test.shape) + " should be the same as that of the original: " + str(rgb_test.shape))

(50000, 32, 32, 3) should be the same as that of the original: (50000, 32, 32, 3)
(10000, 32, 32, 3) should be the same as that of the original: (10000, 32, 32, 3)


In [91]:
print ("There are " + str(len(label_train)) + " classified images in the training set.")
first_five_labels = label_train[0:5]
print ("The first five images in the training set are classified as follows: " + str(first_five_labels))
print ("The result of one-hot-encoding on the first five images yields the following matrix:")
print (to_categorical(first_five_labels, num_classes = 10))

There are 50000 classified images in the training set.
The first five images in the training set are classified as follows: [[6]
 [9]
 [9]
 [4]
 [1]]
The result of one-hot-encoding on the first five images yields the following matrix:
[[0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]]


In [92]:
# One-hot encode the labels
one_hot_label_train = to_categorical(label_train, num_classes = 10)
print("The training labels should be a 50,000 x 10 matrix.  Its shape is: " + str(one_hot_label_train.shape))
one_hot_label_test = to_categorical(label_test, num_classes = 10)
print("The test labels should be a 10,000 x 10 matrix.  Its shape is: " + str(one_hot_label_train.shape))

The training labels should be a 50,000 x 10 matrix.  Its shape is: (50000, 10)
The test labels should be a 10,000 x 10 matrix.  Its shape is: (50000, 10)


In [93]:
# Split the data into training and validation sets
rgb_train_train, rgb_validation, one_hot_label_train_train, one_hot_label_validation = \
  train_test_split(rgb_train, one_hot_label_train, test_size = 0.1, random_state = 77)
print("And the image training set is of size: " + str(rgb_train_train.shape))
print("And the label training set is of size: " + str(one_hot_label_train_train.shape))
print("And the image validation set is of size: " + str(rgb_validation.shape))
print ("And the label validation set is of size: " + str(one_hot_label_validation.shape))

And the image training set is of size: (45000, 32, 32, 3)
And the label training set is of size: (45000, 10)
And the image validation set is of size: (5000, 32, 32, 3)
And the label validation set is of size: (5000, 10)


VGG16 (Visual Geometry Group 16) is a deep convolutional neural network architecture that was developed by the Visual Geometry Group at the University of Oxford. It was proposed by researchers Karen Simonyan and Andrew Zisserman in their paper titled "Very Deep Convolutional Networks for Large-Scale Image Recognition," which was presented at the International Conference on Learning Representations (ICLR) in 2015.

The VGG16 architecture gained significant popularity for its simplicity and effectiveness in image classification tasks. It was one of the pioneering models that demonstrated the power of deeper neural networks for visual recognition tasks.

Key characteristics of the VGG16 architecture:

1. Architecture: VGG16 consists of a total of 16 layers, hence the name "16." These layers are stacked one after another, forming a deep neural network.

2. Convolutional Layers: The main building blocks of VGG16 are the convolutional layers. It primarily uses 3x3 convolutional filters throughout the network, which allows it to capture local features effectively.

3. Max Pooling: After each set of convolutional layers, VGG16 applies max-pooling layers with 2x2 filters and stride 2, which halves the spatial dimensions (width and height) of the feature maps and reduces the number of parameters.

4. Fully Connected Layers: Towards the end of the network, VGG16 has fully connected layers that act as a classifier to make predictions based on the learned features.

5. Activation Function: The network uses the Rectified Linear Unit (ReLU) activation function for all hidden layers, which helps with faster convergence during training.

6. Number of Filters: The number of filters in each convolutional layer is relatively small compared to more recent architectures like ResNet or InceptionNet. However, stacking multiple layers allows VGG16 to learn complex hierarchical features.

7. Output Layer: The output layer consists of 1000 units, corresponding to 1000 ImageNet classes. VGG16 was originally trained on the large-scale ImageNet dataset, which contains millions of images from 1000 different classes.

VGG16 was instrumental in showing that increasing the depth of a neural network can significantly improve its performance on image recognition tasks. However, the main drawback of VGG16 is its high number of parameters, making it computationally expensive and memory-intensive to train. Despite this limitation, VGG16 remains an essential benchmark architecture and has paved the way for even deeper and more efficient models in the field of computer vision, such as ResNet, DenseNet, and EfficientNet.

Here are your tasks:

1. Load [VGG16](https://keras.io/api/applications/vgg/#vgg16-function) as a base model. Make sure to exclude the top layer.
2. Freeze all the layers in the base model. We'll be using these weights as a feature extraction layer to forward to layers that are trainable.

In [111]:
# Load the pre-trained VGG16 model (excluding the top classifier)
# Want the pretrained weights.
base_model = VGG16(include_top = False, weights='imagenet')

In [112]:
# Freeze the layers in the base model
# Thanks Keras documentation: https://keras.io/guides/transfer_learning/
base_model.trainable = False

Now, we'll add some trainable layers to the base model.

1. Using the base model, add a [GlobalAveragePooling2D](https://keras.io/api/layers/pooling_layers/global_average_pooling2d/) layer, followed by a [Dense](https://keras.io/api/layers/core_layers/dense/) layer of length 256 with ReLU activation. Finally, add a classification layer with 10 units, corresponding to the 10 CIFAR-10 classes, with softmax activation.
2. Create a Keras [Model](https://keras.io/api/models/model/) that takes in approproate inputs and outputs.

In [123]:
# Add a global average pooling layer
input_layer = Input(shape = (32, 32, 3))
# KRISTIAN_NOTE - Using this documentation for Transfer Learning overcomes the
# compile-time error where chaining one layer on another (like this: layer_2(layer_1))
# requires an Input layer.  Feed an input layer to the base model, then chain
# pooling_layer on top of that: https://keras.io/guides/transfer_learning/
# Global average pooling just takes an image and returns a single average value over it.
# It is a "flattening" operation in and of itself: https://www.youtube.com/watch?v=gNRVTCf6lvY
pooling_layer = GlobalAveragePooling2D(keepdims = False)(base_model(input_layer))

In [124]:
# Add a fully connected layer with 256 units and ReLU activation
dense_layer = Dense(256, activation = 'relu')(pooling_layer)

In [125]:
# Add the final classification layer with 10 units (for CIFAR-10 classes) and softmax activation
output_layer = Dense(10, activation = 'softmax')(dense_layer)

In [126]:
# Create the fine-tuned model
# KRISTIAN_NOTE - The base model does not have an input layer by default.
# We chained an Input layer to it in the pooling_layer, but we still need
# our input layer to be the explicit input or else we get a size mismatch error
# with our dataset, which consists of 32 x 32 x 3 images rather than a bunch
# of flattened values due to global average pooling.
cifar_model = Model(inputs = input_layer, outputs = output_layer)

In [127]:
cifar_model.summary()

Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_19 (InputLayer)     │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ vgg16 (Functional)              │ (None, 1, 1, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_14     │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,848,586 (56.64 MB)

 Trainable params: 133,898 (523.04 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

With your model complete it's time to train it and assess its performance.

1. Compile your model using an appropriate loss function. Feel free to play around with the optimizer, but a good starting optimizer might be Adam with a learning rate of 0.001.
2. Fit your model on the training data. Use the validation data to print the accuracy for each epoch. Try training for 10 epochs. Note, training can take a few hours so go ahead and grab a cup of coffee.

**Optional**: See if you can implement an [Early Stopping](https://keras.io/api/callbacks/early_stopping/) criteria as a callback function.

In [128]:
# Compile the model
# KRISTIAN_TODO - Look into the other parameters later...
cifar_model.compile(optimizer=Adam(learning_rate = 0.001), loss="categorical_crossentropy")

In [130]:
# Train the model
# Adding the EarlyStopping callback to prematurely end the training if after 3 epochs
# the model fails to improve.  See the docs:
# https://keras.io/api/callbacks/early_stopping/
early_stopping = EarlyStopping(monitor='val_loss', patience=3)
history = cifar_model.fit(rgb_train_train, one_hot_label_train_train, epochs = 1,\
  validation_data = (rgb_validation, one_hot_label_validation),\
  callbacks = [early_stopping])

1407/1407 ━━━━━━━━━━━━━━━━━━━━ 469s 333ms/step - loss: 2.3986 - val_loss: 1.3599


With your model trained, it's time to assess how well it performs on the test data.

1. Use your trained model to calculate the accuracy on the test set. Is the model performance better than random?
2. Experiment! See if you can tweak your model to improve performance.  

In [131]:
# Evaluate the model on the test set
print(history)